# 03 · GSE65391 · RNA_array · batch correction by replicate bridging

Reads `probes.rds` (notebook 02) and `metadata.rds` (notebook 01). Writes
`data/run_artifacts/GSE65391/probes_corrected.rds`.

**The two batches.** The arrays were run in two batches: 838 arrays in batch 1 and 158 in batch 2. Healthy
children are 4% of batch 1 and 25% of batch 2.

**The replicate pairs.** 24 healthy children had the same blood RNA put on two arrays, and GEO marks the
second array as `set = Technical_Replicate`. For 23 of them, one array is in batch 1 and the other in
batch 2. Each pair is the same RNA measured twice, so a difference between the two arrays of a pair is batch
plus measurement noise. The biology is the same.

**Replicate bridging.** For each probe:
1. for each of the 23 pairs, the batch 2 value minus the batch 1 value, on the log2 scale;
2. the mean of those 23 differences is the batch shift of the probe;
3. the shift is added to that probe on every batch 1 array.

For example, if for one probe the batch 2 arrays read on average 0.30 log2 units higher than their batch 1
replicates, 0.30 is added to that probe on all 838 batch 1 arrays. The replicate samples bridge the two
batches. This is the correction the authors applied to the deposited data, as a ratio of means on the raw
scale (GEO data-processing note for GSE65391); on the log2 scale a ratio is a difference. Correction with
reference samples measured in both batches is one of the methods compared in Luo J et al. A comparison of
batch effect removal methods for enhancement of prediction performance using MAQC-II microarray gene
expression data. *Pharmacogenomics J* 2010;10:278-291.

**Why this method here.** The shift is estimated only from the same blood measured twice. A method that
estimates the shift from all samples in each batch, such as ComBat (Johnson WE, Li C, Rabinovic A.
*Biostatistics* 2007;8:118-127), would also see that batch 2 has more healthy children, and would remove part
of the difference between SLE and healthy children as if it were batch.

**Assumptions.** The shift of a probe is the same for every sample (a constant on the log2 scale), and the
shift measured in healthy blood applies to blood from children with SLE.

**Check.** If all 23 pairs estimate the shift, they agree afterwards by construction, which proves nothing.
So the pairs are split at random into two halves: the shift is estimated from one half and tested on the
other, and then the other way round. After correction, the held-out pairs should show no batch difference.

## Load the normalized probes and find the replicate pairs

In [1]:
source("../src/paths.R")
p    <- readRDS(art("GSE65391", "probes.rds"))
meta <- readRDS(art("GSE65391", "metadata.rds"))
E    <- p$E
h <- meta[meta$disease == "Healthy", ]
pairs <- split(rownames(h), paste(h$subject, h$visit))
pairs <- pairs[vapply(pairs, function(pr) setequal(h[pr, "batch"], c("1", "2")), logical(1))]
b1 <- vapply(pairs, function(pr) pr[h[pr, "batch"] == "1"][1], "")
b2 <- vapply(pairs, function(pr) pr[h[pr, "batch"] == "2"][1], "")
c(probes = nrow(E), arrays = ncol(E), replicate_pairs = length(pairs))

probes          arrays replicate_pairs 
          47231             996              23

**Explanation**
- `probes.rds` (notebook 02) holds `E`, the normalized log2 values of all probes and arrays.
- `h` holds the healthy samples, which include the technical replicates.
- `split(rownames(h), paste(h$subject, h$visit))` groups the arrays by child and visit. For example, the two
  arrays of child BAY-H290 at visit 1 form one group.
- `setequal(h[pr, "batch"], c("1", "2"))` is TRUE when a group has one array in each batch; only those groups
  are kept.
- `vapply(pairs, function(pr) ..., logical(1))` applies the function to each group and requires one TRUE or
  FALSE from each; `vapply(..., "")` requires one text value from each.
- `b1` holds the batch 1 array of each pair and `b2` the batch 2 array, in the same order.

## Batch differences before correction

In [2]:
paired_t <- function(D) rowMeans(D) / (apply(D, 1, sd) / sqrt(ncol(D)))
D <- E[, b2] - E[, b1]
c(median_abs_shift = round(median(abs(rowMeans(D))), 3),
  probes_abs_t_above_5 = sum(abs(paired_t(D)) > 5, na.rm = TRUE))

median_abs_shift probes_abs_t_above_5 
                0.09              4209.00

**Explanation**
- `paired_t(D)` gives the paired t statistic of each probe (Student. The probable error of a mean.
  *Biometrika* 1908;6:1-25): the mean difference divided by its standard error, the SD of the differences
  divided by the square root of the number of pairs. `apply(D, 1, sd)` gives the SD of each row.
- `D` holds, for each probe and pair, batch 2 minus batch 1.
- `median_abs_shift`: the median, over probes, of the absolute mean difference, in log2 units.
- `probes_abs_t_above_5` counts the probes with t above 5 in absolute value. With 23 pairs (22 degrees of
  freedom), t = 5 corresponds to a two-sided p of 0.00005. `na.rm = TRUE` skips probes that are identical in
  every pair.

**Result.** Before correction the two batches differ: the median absolute shift is 0.09 log2 units, and
4,209 of the 47,231 probes have a t statistic above 5 in absolute value.

## Check: estimate on half the pairs, test on the other half

In [3]:
set.seed(SEED)
half <- sample(rep(1:2, length.out = ncol(D)))
check <- do.call(rbind, lapply(1:2, function(k) {
  est  <- D[, half != k, drop = FALSE]
  held <- D[, half == k, drop = FALSE]
  t_after <- (rowMeans(held) - rowMeans(est)) /
             sqrt(apply(held, 1, var) / ncol(held) + apply(est, 1, var) / ncol(est))
  data.frame(held_out_half = k, pairs_held_out = ncol(held), pairs_estimating = ncol(est),
             median_abs_shift_before = round(median(abs(rowMeans(held))), 3),
             median_abs_shift_after  = round(median(abs(rowMeans(held) - rowMeans(est))), 3),
             probes_abs_t_above_5_before = sum(abs(paired_t(held)) > 5, na.rm = TRUE),
             probes_abs_t_above_5_after  = sum(abs(t_after) > 5, na.rm = TRUE))
}))
check
c(expected_by_chance_after = round(nrow(D) * 2 * pt(-5, df = ncol(D) - 2)))

held_out_half,pairs_held_out,pairs_estimating,median_abs_shift_before,median_abs_shift_after,probes_abs_t_above_5_before,probes_abs_t_above_5_after
<int>,<int>,<int>,<dbl>,<dbl>,<int>,<int>
1,12,11,0.095,0.059,1987,2
2,11,12,0.093,0.059,1489,2


expected_by_chance_after 
                       3

**Explanation**
- `sample(rep(1:2, length.out = 23))` assigns each pair at random to half 1 or half 2: 12 and 11 pairs.
  `set.seed(SEED)` makes the split repeatable.
- For each half `k`:
  - `est` holds the differences of the pairs that estimate the shift: `half != k`;
  - `held` holds the differences of the held-out pairs;
  - the shift estimated from `est` is `rowMeans(est)`; `rowMeans(held) - rowMeans(est)` is what remains of the
    held-out batch difference after the correction.
- `t_after` divides what remains by its standard error. The standard error has two parts: the error of the
  held-out mean, `var(held) / n_held`, and the error of the estimated shift, `var(est) / n_est`. Leaving out
  the second part would make t too large, because the shift is estimated from only 11 or 12 pairs. This is
  Welch's two-sample t statistic (Welch BL. The generalization of "Student's" problem when several different
  population variances are involved. *Biometrika* 1947;34:28-35).
- Before correction, the t statistic is the paired t of the held-out pairs, as in the step above.
- `expected_by_chance_after` is the number of probes that would pass t above 5 in absolute value if no batch
  difference were left: the number of probes times the two-sided p for t = 5 with 21 degrees of freedom.
  `pt(-5, df)` gives the probability below -5; `2 *` counts both sides.

**Result.** On the held-out pairs, the probes with a t statistic above 5 in absolute value fall from 1,987 and
1,489 before correction to 2 and 2 after; 3 would be expected by chance if no batch difference were left.
The remaining median absolute difference, 0.059 log2 units in both halves, is the size of the measurement
noise of 11 or 12 pairs. The correction removes the batch difference, as far as this check can detect.

## Correct batch 1 with all 23 pairs

In [4]:
shift <- rowMeans(D)
E_corrected <- E
in_batch1 <- meta[colnames(E), "batch"] == "1"
E_corrected[, in_batch1] <- E[, in_batch1] + shift
D_after <- E_corrected[, b2] - E_corrected[, b1]
cc <- cor(E_corrected[p$detected, b1])
c(max_abs_mean_difference_after = signif(max(abs(rowMeans(D_after))), 2),
  replicate_pair_correlation = round(mean(diag(cor(E_corrected[p$detected, b1], E_corrected[p$detected, b2]))), 3),
  different_children_correlation = round(mean(cc[upper.tri(cc)]), 3))

max_abs_mean_difference_after     replicate_pair_correlation 
                      8.50e-16                       9.81e-01 
different_children_correlation 
                      9.56e-01

**Explanation**
- `shift` is each probe's mean difference over all 23 pairs.
- `E_corrected[, in_batch1] <- E[, in_batch1] + shift` adds each probe's shift to that probe on every batch 1
  array. R adds the vector `shift` down each column, one value per probe.
- `max_abs_mean_difference_after`: over all 23 pairs the mean difference is now 0 for every probe, by
  construction; the value printed is rounding error.
- The last two values compare, over the detected probes, the correlation of the two arrays of a replicate
  pair with the correlation of arrays from different children in batch 1. Replicates should correlate more.
  `diag()` keeps each array with its own replicate; `cc[upper.tri(cc)]` keeps each pair of different arrays
  once.

## Drop the technical replicates and save

**Result.** After correction, the arrays of a replicate pair correlate at 0.981 and arrays of different
children at 0.956.

In [5]:
keep <- meta[colnames(E_corrected), "set"] != "Technical_Replicate"
saveRDS(list(E = E_corrected[, keep], detected = p$detected, quality_ok = p$quality_ok, entrez = p$entrez,
             meta = meta[colnames(E_corrected)[keep], ]),
        art("GSE65391", "probes_corrected.rds"))
table(disease = meta[colnames(E_corrected)[keep], "disease"])

disease
Healthy     SLE 
     48     924 

**Explanation**
- The technical replicates are second arrays of samples already in the table. After the correction they are
  no longer needed and are dropped.
- `probes_corrected.rds` keeps the corrected values of all probes for the remaining arrays, the probe
  information from notebook 02, and the sample table. Notebook 03b collapses the probes to genes; notebook 06d
  uses the probes directly.

**Result.** 972 arrays remain: 48 healthy, 924 SLE.